
<b><font size = 2><span style="color:#00a896"> 💬 Text to Image generation (Stable Diffusion)🖼️  </span></font></b>  

<b><font size = 2><span style="color:#00a896">Created By Burhanuddin Latsaheb </span></font> </b> 


# <center><font size = 6><span style="color:#05668d"> 💬Text to Image generation🖼️ (Stable Diffusion) </span></font></center>  

## <center><font size =4><span style="color:#05668d"> If you find this notebook useful,support with an upvote👍👍 </span></font></center>


In [ ]:
!pip install diffusers==0.3.0 --q
!pip install transformers scipy ftfy --q
!pip install "ipywidgets>=7,<8" --q
import IPython.display 


# <center><font size = 3><span style="color:#a8dadc"> <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:200%;text-align:center;border-radius:100px 10px;">INTRODUCTION </p>   </span></font></center>

<font size = 5><span style="color:#05668d">Notebook Overview : </span></font>

 <font size = 3><span style="color:#3A3E59"> This notebook contains:  </span></font>
    1. <font size = 3><span style="color:#3A3E59"> Building our own pipeline for stable diffusion </span></font>
    2. <font size = 3><span style = "color:#3A3E59">Using a pretrained pipeline for stable diffusion </span></font>
*  <font size = 3><span style="color:#3A3E59">The pretrained model, pipelines and schedulers are available on <a href = https://huggingface.co/CompVis/stable-diffusion-v1-4>hugging face</a></span></font>
*  <font size = 3><span style="color:#3A3E59">The  pretrained models are loaded using the `Diffusers` library and they are denoised using the `Pytorch` library</span></font>



<a id = "top"></a>
# <center><font size = 3><span style="color:#a8dadc"> <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:200%;text-align:center;border-radius:100px 10px;">Table of Contents </p>   </span></font></center>

- [1. Imports](#imports)
- [2. Hyperparameters](#hyperparameters)
- [3. Helper Functions](#helper_functions)
- [4. Pipeline for Stable Diffusion ](#pipeline)
  * [4.1. Loading the pretrained models](#pretrained)
  * [4.2. Scheduler](#scheduler)
  * [4.3. Building first image](#building_1)
      + [4.3.1 Encoding the image](#encode_1)
      + [4.3.2 Decoding the image](#decode_1)
      + [4.3.3 Visualizing the image](#visualize_1)
 * [4.4. Second image](#building_2)
      + [4.4.1 Encoding the image](#encode_2)
      + [4.4.2 Decoding the image](#decode_2)
      + [4.4.3 Visualizing the image](#visualize_2)
- [5. Pretrained Pipeline for Stable Diffusion](#pipeline_2)
 * [5.1 Visualizing the images](#visualize)   


<font size = 5><span style="color:#05668d">Steps before running the notebook : </span></font>

1. <font size = 3><span style="color:#3A3E59"> You need to create a token from <a href=https://huggingface.co/> here </a></span></font>

In [ ]:
IPython.display.Image("../input/markdown/1.png")

In [ ]:
IPython.display.Image("../input/markdown/2.png")

 2.  <font size = 3><span style="color:#3A3E59"> Copy the token from <a href = https://huggingface.co/settings/tokens> here</a> then add it in the secrets section of the Add-ons section set the label to `Hugging_id` and paste yout token in value.  </span></font>

In [ ]:
IPython.display.Image("../input/markdown/3.png")

In [ ]:
IPython.display.Image("../input/markdown/4.png")


<a id = "imports"></a>
<a id="1"></a>
# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">1. IMPORTS 📂</p>
#### [Top ↑](#top)

In [ ]:

import gc
import torch
from PIL import Image
import IPython.display 
from torch import autocast
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import StableDiffusionPipeline
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers import LMSDiscreteScheduler , PNDMScheduler


user_secrets = UserSecretsClient()
Hugging_face  = user_secrets.get_secret("Hugging_id")


<a id = "hyperparameters"></a>
# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">2. HYPERPARAMETERS 🏹</p>
#### [Top ↑](#top)

In [ ]:
class config : 
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    HEIGHT = 512                        
    WIDTH = 512                         
    NUM_INFERENCE_STEPS = 500            
    GUIDANCE_SCALE = 7.5                
    GENERATOR = torch.manual_seed(48)   
    BATCH_SIZE = 1
    


<a id = "helper_functions"></a>
# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">3.HELPER FUNCTIONS</p>
#### [Top ↑](#top)

In [ ]:
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols
    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid


<a id = "pipeline"></a>
# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4. PIPELINE</p>
#### [Top ↑](#top)

<a id = "pretrained"></a>
<p style="background-color:#f0f3bd;font-family:newtimeroman;color:#028090;font-size:140%;text-align:center;border-radius:200px 10px;">4.1. Loading the pretrained models</p>

* <font size = 3><span style="color:#3A3E59"> The model we are going to use is `CompVis/stable-diffusion-v1-4` the model card can be found <a href =https://huggingface.co/CompVis/stable-diffusion-v1-4>here</a> </span></font>
* <font size = 3><span style="color:#3A3E59"> We are going to load 
     `Variable auto encoder`,
     `Tokenizer`,
     `Text encoder`  and
     `Unet`</span></font>
     
* <font size = 3><span style="color:#3A3E59">Stable Diffusion during inference</span></font>
 
#### [Top ↑](#top)

In [ ]:
IPython.display.Image("../input/markdown/stable_diffusion.png")


In [ ]:
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae", use_auth_token=Hugging_face)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet", use_auth_token=Hugging_face)
vae = vae.to(config.DEVICE)
text_encoder = text_encoder.to(config.DEVICE)
unet = unet.to(config.DEVICE) 


In [ ]:
print(f'\033[94mTokenizer, Text Encoder, VAE, Unet are loaded !!')


<a id = "scheduler"></a>
<p style="background-color:#f0f3bd;font-family:newtimeroman;color:#028090;font-size:140%;text-align:center;border-radius:200px 10px;">4.2. Scheduler</p>

<font size = 5><span style="color:#F60195"> </span></font>
* <font size = 3><span style="color:#3A3E59"> I am going to use the K - LMS Scheduler</span></font>
* <font size = 3><span style="color:#3A3E59"> The default scheduler is PNDM scheduler </span></font>
* <font size = 3><span style="color:#3A3E59"> Some other schedulers are DDIM  ,DDPM and <a href = https://github.com/huggingface/diffusers/tree/main/src/diffusers/schedulers> some more </a></span></font>


#### [Top ↑](#top)

In [ ]:
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)
print(f'\033[94mThe scheduler loaded is K-LMS Sceheduler')


<a id = "building_1"></a>

<p style="background-color:#f0f3bd;font-family:newtimeroman;color:#028090;font-size:140%;text-align:center;border-radius:200px 10px;">4.3. FIRST IMAGE</p>

* <font size = 3><span style="color:#3A3E59">tokenising the prompt</span></font>
* <font size = 3><span style="color:#3A3E59">text embeddings </span></font>
* <font size = 3><span style="color:#3A3E59">latent intialization</span></font>

#### [Top ↑](#top)

<font size = 3><span style="color:#3A3E59">The text we are going to visualize is :</span></font>
<div style="background-color:rgba(0, 0, 0, 0.0470588); text-align:center; vertical-align: middle; padding:40px 0;">
<font size = 3><span style="color:#3A3E59"> 
    a curious explorer discovers a massive sprawling underground city in a huge cave system. city has churches, european - style buildings and big towers, and is really far away, illuminated by dim ambient lighting. waterfalls are flowing between different levels of the city. award winning digital art, concept art, breathtaking, imaginative, detailed</span></font>
</div>

<font size = 3><span style="color:#3A3E59">The text can be visualized as  : </span></font>

In [ ]:
IPython.display.Image("../input/markdown/cave.png")

In [ ]:
prompt = ["a curious explorer discovers a massive sprawling underground city in a huge cave system. city has churches, european - style buildings and big towers, and is really far away, illuminated by dim ambient lighting. waterfalls are flowing between different levels of the city. award winning digital art, concept art, breathtaking, imaginative, detailed., 8k"]

In [ ]:
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
max_length = text_input.input_ids.shape[-1]
with torch.no_grad():
      text_embeddings = text_encoder(text_input.input_ids.to(config.DEVICE))[0]
uncond_input = tokenizer(
    [""] * config.BATCH_SIZE, padding="max_length", max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
      uncond_embeddings = text_encoder(uncond_input.input_ids.to(config.DEVICE))[0]   
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
print(f'\033[94mText Embeddings shape: {text_embeddings.shape}')


In [ ]:
latents = torch.randn(
  (config.BATCH_SIZE, unet.in_channels, config.HEIGHT // 8, config.WIDTH // 8),
  generator=config.GENERATOR,
)
latents = latents.to(config.DEVICE)

print(f'\033[94mLatent shape: {latents.shape}')


<a id = "encode_1"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.3.1 Encoding the image</p>

#### [Top ↑](#top)

In [ ]:
scheduler.set_timesteps(config.NUM_INFERENCE_STEPS)
latents = latents * scheduler.sigmas[0]

In [ ]:

with autocast(config.DEVICE):
      for i, t in tqdm(enumerate(scheduler.timesteps)):
        
        latent_model_input = torch.cat([latents] * 2)
        sigma = scheduler.sigmas[i]
        latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)

        with torch.no_grad():
              noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + config.GUIDANCE_SCALE * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, i, latents).prev_sample

<a id = "decode_1"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.3.2 Decoding the image</p>

#### [Top ↑](#top)

In [ ]:
latents = 1 / 0.18215 * latents

with torch.no_grad():
  image = vae.decode(latents).sample
print(f'\033[94mImage shape: {image.shape}')


<a id = "visualize_1"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.3.3 Visualizing the image</p>

#### [Top ↑](#top)


In [ ]:
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

<a id = "building_2"></a>
<p style="background-color:#f0f3bd;font-family:newtimeroman;color:#028090;font-size:140%;text-align:center;border-radius:200px 10px;">4.4. SECOND IMAGE</p>

* <font size = 3><span style="color:#3A3E59">tokenising the prompt</span></font>
* <font size = 3><span style="color:#3A3E59">text embeddings </span></font>
* <font size = 3><span style="color:#3A3E59">latent intialization</span></font>


<font size = 3><span style="color:#3A3E59">The text we are going to visualize is :</span></font>
<div style="background-color:rgba(0, 0, 0, 0.0470588); text-align:center; vertical-align: middle; padding:40px 0;">
<font size = 3><span style="color:#3A3E59"> 
   a ultradetailed beautiful panting of a stylish woman with a sword, by conrad roset, greg rutkowski and makoto shinkai, trending on artstation</span></font>
</div>


<font size = 3><span style="color:#3A3E59">The text can be visualized as  : </span></font>


In [ ]:
IPython.display.Image("../input/markdown/girl.png")

In [ ]:
prompt = ["a ultradetailed beautiful panting of a stylish woman with a sword, by conrad roset, greg rutkowski and makoto shinkai, trending on artstation , 8k"]

In [ ]:
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
max_length = text_input.input_ids.shape[-1]
with torch.no_grad():
      text_embeddings = text_encoder(text_input.input_ids.to(config.DEVICE))[0]
uncond_input = tokenizer(
    [""] * config.BATCH_SIZE, padding="max_length", max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
      uncond_embeddings = text_encoder(uncond_input.input_ids.to(config.DEVICE))[0]   
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
print(f'\033[94mText Embeddings shape: {text_embeddings.shape}')


In [ ]:
latents = torch.randn(
  (config.BATCH_SIZE, unet.in_channels, config.HEIGHT // 8, config.WIDTH // 8),
  generator=config.GENERATOR,
)
latents = latents.to(config.DEVICE)

print(f'\033[94mLatent shape: {latents.shape}')


<a id = "encode_2"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.4.1 Encoding the image</p>

#### [Top ↑](#top)



In [ ]:
scheduler.set_timesteps(config.NUM_INFERENCE_STEPS)
latents = latents * scheduler.sigmas[0]

In [ ]:

with autocast(config.DEVICE):
      for i, t in tqdm(enumerate(scheduler.timesteps)):
        
        latent_model_input = torch.cat([latents] * 2)
        sigma = scheduler.sigmas[i]
        latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)

        with torch.no_grad():
              noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + config.GUIDANCE_SCALE * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, i, latents).prev_sample

<a id = "decode_2"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.4.2 Decoding the image</p>

#### [Top ↑](#top)



In [ ]:
latents = 1 / 0.18215 * latents

with torch.no_grad():
  image = vae.decode(latents).sample
print(f'\033[94mImage shape: {image.shape}')


<a id = "visualize_2"></a>

<p style="background-color:#02c39a;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">4.4.3 Visualizing the image</p>

#### [Top ↑](#top)


In [ ]:
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0].save("img2.jpg")
pil_images[0]

In [ ]:
del latents
del vae
del text_encoder
del unet
gc.collect()

<a id = "pipeline_2"></a>
# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">5. PRETRAINED PIPELINE FOR STABLE DIFFUSION</p>
#### [Top ↑](#top)


* <font size = 3><span style="color:#3A3E59">StableDiffusionPipeline is an end-to-end inference pipeline that we can use to generate images from text with just a few lines of code.
</span></font>


In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", revision="fp16", torch_dtype=torch.float16, use_auth_token=Hugging_face)  
pipe = pipe.to(config.DEVICE)
print(f'\033[94mStable Diffusion Pipeline created !!!')


<a id = "visualize"></a>

<p style="background-color:#f0f3bd;font-family:newtimeroman;color:#028090;font-size:140%;text-align:center;border-radius:200px 10px;">5.1 Visualizing the images</p>
<font size = 3><span style="color:#3A3E59">The text we are going to visualize is :</span></font>
<div style="background-color:rgba(0, 0, 0, 0.0470588); text-align:center; vertical-align: middle; padding:40px 0;">
<font size = 3><span style="color:#3A3E59"> 
  
futuristic synthwave city, retro sunset, crystals, spires, volumetric lighting, studio Ghibli style, rendered in unreal engine with clean</span></font>
</div>


In [ ]:
num_images = 4
prompt = ["futuristic synthwave city, retro sunset, crystals, spires, volumetric lighting, studio Ghibli style, rendered in unreal engine with clean details --ar 9:16 --hd --q 2"] * num_images
with autocast("cuda"):
  images = pipe(prompt , num_inference_steps=200).images

grid = image_grid(images, rows=2, cols=2)
grid

<font size = 3><span style="color:#3A3E59">The text we are going to visualize is :</span></font>
<div style="background-color:rgba(0, 0, 0, 0.0470588); text-align:center; vertical-align: middle; padding:40px 0;">
<font size = 3><span style="color:#3A3E59"> 
   postapocalyptic city turned to fractal glass, shiny : : close shot : : 3 5 mm, realism, octane render, 8 k, exploration, cinematic, trending on artstation, realistic, 3 5 mm camera, unreal engine, hyper detailed, photo - realistic maximum detail, volumetric light, moody cinematic epic concept art, realistic matte painting, hyper photorealistic, concept art, volumetric light, cinematic epic, octane render, 8 k, corona render, movie concept art, octane render, 8 k, corona render, cinematic, trending on artstation, movie concept art, cinematic composition, ultra - detailed, realistic, hyper - realistic, volumetric lighting, 8 k</span></font>
</div>


In [ ]:
num_images = 4
prompt =["postapocalyptic city turned to fractal glass, shiny : : close shot : : 3 5 mm, realism, octane render, 8 k, exploration, cinematic, trending on artstation, realistic, 3 5 mm camera, unreal engine, hyper detailed, photo - realistic maximum detail, volumetric light, moody cinematic epic concept art, realistic matte painting, hyper photorealistic, concept art, volumetric light, cinematic epic, octane render, 8 k, corona render, movie concept art, octane render, 8 k, corona render, cinematic, trending on artstation, movie concept art, cinematic composition, ultra - detailed, realistic, hyper - realistic, volumetric lighting, 8 k"] * num_images
with autocast("cuda"):
  images = pipe(prompt , num_inference_steps=200).images

grid = image_grid(images, rows=2, cols=2)
grid

<font size = 3><span style="color:#3A3E59">The text we are going to visualize is :</span></font>
<div style="background-color:rgba(0, 0, 0, 0.0470588); text-align:center; vertical-align: middle; padding:40px 0;">
<font size = 3><span style="color:#3A3E59"> 
   Cybernetic cloaked anime character concept design, dynamic pose, fantasy anime, dark, bejewelled and encrusted technological royal cloak, powerful aggressive sword stance, biological human face, iridescent, dark and intricate, Greg Rutkowski, Makoto Shinkai, anime CGI, animated, animation, artgerm, artstation, digital illustration</span></font>
</div>


In [ ]:
num_images = 4
prompt =["Cybernetic cloaked anime character concept design, dynamic pose, fantasy anime, dark, bejewelled and encrusted technological royal cloak, powerful aggressive sword stance, biological human face, iridescent, dark and intricate, Greg Rutkowski, Makoto Shinkai, anime CGI, animated, animation, artgerm, artstation, digital illustration, 8k"] * num_images
with autocast("cuda"):
  images = pipe(prompt , num_inference_steps=200).images

grid = image_grid(images, rows=2, cols=2)
grid

# <p style="background-color:#028090;font-family:newtimeroman;color:#f0f3bd;font-size:140%;text-align:center;border-radius:200px 10px;">6. REFERENCES 📜</p>

#### [Top ↑](#top)

* <a href = https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb> 
https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb</a>
* <a href = https://arxiv.org/abs/2112.10752> https://arxiv.org/abs/2112.10752</a>
* <a href = https://arxiv.org/pdf/2205.11487.pdf> https://arxiv.org/pdf/2205.11487.pdf</a>


<center><b><font size = 3><span style="color:#2F4F4F"> Thank You for reading 😊</span></font></b></center>

<center><b><font size = 3><span style="color:#2F4F4F"> If you have any suggestions or feeback, please let me know</span></font></b></center>